# 01 · The Hugging Face Hub and `pipeline()`

After this notebook you can find a model on the Hub, read its card and licence, run it locally in three lines with `pipeline()`, download exactly the files you need, pin the version you tested, and explain the security traps (pickle files, `trust_remote_code`, tokens).

**Time:** ~25 min · **Runs:** offline on CPU in about a minute (the cells that read Hub metadata use the network when you have it and skip politely when you don't) · **Needs:** basic Python; [PyTorch essentials](../04_numpy_pandas_pytorch/03_pytorch_essentials.ipynb) helps

## Why this matters in interviews

- "What is Hugging Face, exactly?" comes up in almost every GenAI interview. A strong answer names the three parts (the Hub, the libraries, hosted inference) and then mentions supply-chain risk without being asked.
- Interviewers for self-hosted or regulated projects want to know you can pick an open model on purpose: its licence, its size, base or instruct, and whether it ships safetensors.
- `pipeline()` is the fastest way to prove an idea works. Being able to say what it does inside (tokenise → forward pass → post-process) shows you understand the model and not just the wrapper.
- Pinning revisions, caching models and running offline are what separate a laptop demo from a reproducible service.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `tl27` | What is Hugging Face, exactly? |
| `sf27` | What is the supply-chain risk in an LLM stack? |
| `sf17` | What are the copyright and licensing risks? |
| `fo09` | What is the difference between a base model, an instruct model and a chat model? |
| `fo17` | What is the difference between an encoder, a decoder and an encoder-decoder model? |
| `in19` | When would you self-host a model instead of using an API? |

In [ ]:
import os, time, json, struct, pickle, tempfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import transformers, huggingface_hub
from transformers import pipeline

transformers.utils.logging.set_verbosity_error()   # hide info/deprecation chatter so the outputs stay readable
transformers.utils.logging.disable_progress_bar()  # no "Loading weights" bars
torch.manual_seed(0)
print(f"transformers {transformers.__version__} | huggingface_hub {huggingface_hub.__version__} | torch {torch.__version__}")

## 1. What "Hugging Face" means

**In plain English:** Hugging Face is where open models live. It works like GitHub and an app store combined: anyone can publish a model or a dataset, and the company's Python libraries download and run them in a few lines.

When someone says "Hugging Face", they can mean any of three things, so say which one you mean:

| Part | What it is | You use it for |
|---|---|---|
| **The Hub** (huggingface.co) | Git repositories for **models**, **datasets** and **Spaces** (small hosted demo apps, usually Gradio or Streamlit) | finding, versioning and sharing weights and data |
| **The libraries** | `transformers`, `huggingface_hub`, `datasets`, `sentence-transformers`, `peft`, `trl`, `accelerate`, `safetensors`, `tokenizers` | running and training models on your own hardware |
| **Hosted inference** | Inference Providers (a router to partner GPU clouds) and Inference Endpoints (your own dedicated deployment) | calling a model without owning a GPU |

A **model repo** is a git repo. The large files are stored with Git LFS or Xet. A typical one holds:

| File | What it is |
|---|---|
| `config.json` | the architecture: layer count, hidden size, vocabulary size, `model_type` |
| `model.safetensors` | the weights (large repos split them into shards plus an index file) |
| `tokenizer.json`, `tokenizer_config.json` | the tokenizer and its special tokens (and often the chat template) |
| `generation_config.json` | default generation settings (eos token, sometimes temperature) |
| `README.md` | the **model card**: licence, intended use, training data, evaluation |

Every push creates a **commit**. `main` is a branch that moves; a commit sha never changes. Section 6 shows why that matters.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

def try_hub(fn, *args, **kwargs):
    """Call the Hub API; return None (and say so) when offline or the Hub is unreachable."""
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print(f"[skipped {fn.__name__}: offline or Hub unreachable ({type(e).__name__})]")
        return None

info = try_hub(api.model_info, "HuggingFaceTB/SmolLM2-135M-Instruct", timeout=10)
if info:
    print("repo        :", info.id)
    print("commit sha  :", info.sha)
    print("task        :", info.pipeline_tag, "| library:", info.library_name, "| gated:", info.gated)
    print("licence     :", info.card_data.license, "| base model:", info.card_data.get("base_model"))
    print("parameters  :", info.safetensors.parameters if info.safetensors else "n/a")
    print("popularity  :", info.downloads, "downloads (last 30 days),", info.likes, "likes")
    print("top files   :", [s.rfilename for s in info.siblings if "/" not in s.rfilename])
    print("other files :", len([s for s in info.siblings if "/" in s.rfilename]), "in sub-folders (ONNX exports, training logs, ...)")

The metadata already answers most of the questions you should ask before you use a model:

- **Licence**: can you use it commercially? (next section)
- **Base or instruct**: `base_model` points to `SmolLM2-135M`, the *base* model this *instruct* model was fine-tuned from. A base model only continues text. An instruct (chat) model was further trained on conversations, so it follows instructions and expects its chat template (`fo09`).
- **Size**: 134.5 M parameters stored in bf16 (2 bytes each), so the weights are about 270 MB.
- **Formats**: safetensors, plus ONNX exports for other runtimes.

Downloads and likes tell you a model is popular, not that it is good for your task. You still evaluate it on your own data.

In [ ]:
family = try_hub(lambda: list(api.list_models(search="SmolLM2", sort="downloads", limit=6)))
if family:
    print(f"{'model':45s} {'task':18s} downloads")
    for m in family:
        print(f"{m.id:45s} {str(m.pipeline_tag):18s} {m.downloads:>10,}")

One family, several sizes, and each size comes as a base model and as an `-Instruct` model. The usual approach is to start with the smallest size that passes your evaluation, because every doubling of parameters roughly doubles memory and per-token cost.

## 2. Model cards and licences

The model card is `README.md`. It starts with a YAML header (licence, datasets, base model, tags) that the Hub reads, followed by prose: intended use, limitations, training data and evaluation. `huggingface_hub.ModelCard` parses both parts.

In [ ]:
from huggingface_hub import ModelCard

try:
    card = ModelCard.load("sentence-transformers/all-MiniLM-L6-v2")   # README.md is already in the local cache
    meta = card.data.to_dict()
    print("licence :", meta.get("license"))
    print("task    :", meta.get("pipeline_tag"), "| library:", meta.get("library_name"))
    print("trained on", len(meta.get("datasets", [])), "datasets, e.g.", meta.get("datasets", [])[:3])
    print("--- first lines of the prose ---")
    print("\n".join(card.text.strip().splitlines()[:3]))
except Exception as e:
    print("Could not load the model card (offline and not cached?):", type(e).__name__)

"Open weights" does not mean "open source", and it does not mean you can use the model however you like (`sf17`). Read the licence before you build on a model:

| Licence family | Commercial use? | Watch out for |
|---|---|---|
| `apache-2.0`, `mit` | yes | keep the licence and attribution notices |
| Llama community licences | yes, with conditions | an acceptable-use policy, attribution, and a separate licence above 700 M monthly active users; gated on the Hub |
| Gemma terms of use | yes, with conditions | a prohibited-use policy that carries over to fine-tunes |
| `cc-by-nc-4.0` and other "NC" licences | **no** | common for research checkpoints; a fine-tune of an NC model is still NC |
| OpenRAIL family | usually yes | use-based restrictions written into the licence |

Two more things to check: whether the **training data** licence allows your use, and whether you have the rights to the data you fine-tune on. The licence can also differ between sizes of the same family, so check the exact repo.

## 3. `pipeline()`: three lines from text to prediction

**In plain English:** `pipeline(task, model=...)` downloads the model and its tokenizer once, then turns raw inputs into readable answers. Inside it always does the same three steps:

In [ ]:
from matplotlib.patches import FancyBboxPatch

def draw_pipeline(steps, title=None):
    """Boxes joined by arrows, left to right."""
    fig, ax = plt.subplots(figsize=(min(2.3 * len(steps), 11), 1.7))
    for i, s in enumerate(steps):
        x = i * 2.3
        ax.add_patch(FancyBboxPatch((x, 0), 1.9, 1, boxstyle="round,pad=0.06", fc="#e7f0ff", ec="#3b5bdb"))
        ax.text(x + 0.95, 0.5, s, ha="center", va="center", fontsize=9)
        if i < len(steps) - 1:
            ax.annotate("", xy=(x + 2.28, 0.5), xytext=(x + 1.98, 0.5), arrowprops=dict(arrowstyle="->", lw=1.6))
    ax.set_xlim(-0.2, len(steps) * 2.3); ax.set_ylim(-0.25, 1.25); ax.axis("off")
    if title: ax.set_title(title)
    plt.show()

draw_pipeline(["raw text", "preprocess\n(tokenizer)", "model forward\n(logits)", "postprocess\n(softmax, labels)", "label + score"],
              title="What pipeline('sentiment-analysis') does")

In [ ]:
t0 = time.perf_counter()
sentiment = pipeline("sentiment-analysis",
                     model="distilbert/distilbert-base-uncased-finetuned-sst-2-english", device="cpu")
print(f"loaded in {time.perf_counter() - t0:.1f}s -> {type(sentiment).__name__}, task = {sentiment.task!r}")

reviews = [
    "The battery lasts two days and the screen is gorgeous.",
    "Support never answered my emails. Avoid.",
    "The movie was not bad.",
    "Great, it broke after two days.",       # sarcasm
    "The package arrived on Tuesday.",       # neutral statement
]
results = sentiment(reviews, batch_size=8)   # a list in, a list out, run in batches
for text, r in zip(reviews, results):
    print(f"{r['label']:>8}  {r['score']:.3f}  {text}")

assert results[0]["label"] == "POSITIVE" and results[1]["label"] == "NEGATIVE"
assert results[2]["label"] == "POSITIVE"    # it handles the simple negation "not bad"

Read the scores critically:

- The obvious cases are right, and the model copes with negation ("not bad" comes out positive).
- **Sarcasm** fools it: "Great, it broke..." comes out positive.
- **There is no "neutral" class.** This model was fine-tuned on SST-2, which has only two labels, so a neutral sentence still gets one of them with a high score.
- The **score is a softmax probability**, not a calibrated confidence. A high score on a neutral sentence is the model being sure within its two-label world, not the model being right.

`top_k=None` returns the probability of every label, which is what you need for thresholds (for example, route to a human when neither label clears 0.9).

In [ ]:
all_scores = sentiment(reviews, top_k=None)
p_pos = [next(d["score"] for d in row if d["label"] == "POSITIVE") for row in all_scores]
for row in all_scores:
    assert abs(sum(d["score"] for d in row) - 1) < 1e-5     # the softmax over the 2 labels sums to 1

fig, ax = plt.subplots(figsize=(9, 3))
ax.barh(range(len(reviews)), p_pos, color=["#2f9e44" if p > 0.5 else "#e03131" for p in p_pos])
ax.set_yticks(range(len(reviews)), [r[:38] + ("..." if len(r) > 38 else "") for r in reviews])
ax.axvline(0.5, color="gray", ls="--", lw=1)
ax.invert_yaxis(); ax.set_xlim(0, 1)
ax.set_xlabel("P(POSITIVE)"); ax.set_title("Sentiment pipeline: probability of POSITIVE per review")
plt.show()

### What the pipeline did, by hand

Interviewers like "what does `pipeline` do under the hood?". Here are the same three steps written out with the tokenizer and model the pipeline already loaded. The asserts check that we get exactly the pipeline's answers.

In [ ]:
tok, clf_model = sentiment.tokenizer, sentiment.model
batch = tok(reviews, padding=True, truncation=True, return_tensors="pt")     # 1. preprocess
with torch.no_grad():
    logits = clf_model(**batch).logits                                         # 2. forward pass
probs = logits.softmax(dim=-1)                                                 # 3. postprocess
labels = [clf_model.config.id2label[i] for i in probs.argmax(dim=-1).tolist()]

print("input_ids shape :", tuple(batch["input_ids"].shape), "(reviews x tokens, padded to the longest)")
print("logits[0]       :", logits[0].numpy().round(2), "->", clf_model.config.id2label)
print("labels          :", labels)
assert labels == [r["label"] for r in results]
assert np.allclose(probs.max(dim=-1).values.numpy(), [r["score"] for r in results], atol=1e-5)
print("manual tokenizer -> model -> softmax matches pipeline() exactly")

### Feature extraction: vectors instead of labels

`"feature-extraction"` returns the model's last hidden state: **one vector per token**, not one per sentence. To get a sentence embedding you pool those vectors, usually by averaging them. The [sentence-transformers notebook](04_sentence_transformers_embeddings_and_cross_encoders.ipynb) does this properly, with batching and normalisation.

In [ ]:
extractor = pipeline("feature-extraction", model="sentence-transformers/all-MiniLM-L6-v2", device="cpu")
sentences = ["How do I reset my password?", "I forgot my login credentials.", "What time does the museum open?"]

token_vectors = [np.array(extractor(s))[0] for s in sentences]            # each: (n_tokens, 384)
print("per-token output shapes:", [v.shape for v in token_vectors])
sent_vecs = np.stack([v.mean(axis=0) for v in token_vectors])             # mean pooling -> (3, 384)
sent_vecs /= np.linalg.norm(sent_vecs, axis=1, keepdims=True)
sims = sent_vecs @ sent_vecs.T
print(f"cos(password, credentials) = {sims[0, 1]:.3f}")
print(f"cos(password, museum)      = {sims[0, 2]:.3f}")
assert sims[0, 1] > sims[0, 2]    # same meaning, different words -> closer

### Text generation with an instruct model

Two families have appeared so far (`fo17`). DistilBERT and MiniLM are **encoders**: they read the whole input at once and produce labels or vectors. SmolLM2 is a **decoder**: it writes text one token at a time. If you pass a list of chat messages, the pipeline applies the model's chat template for you ([notebook 02](02_tokenizers_models_and_generate.ipynb) prints that template).

We load it in `float32`. In transformers v5, `from_pretrained` and `pipeline` default to `dtype="auto"`, which keeps the checkpoint's own dtype (bf16 here). That halves the memory, but on many laptop CPUs bf16 matrix maths is slower than fp32.

In [ ]:
t0 = time.perf_counter()
chat = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-135M-Instruct", dtype=torch.float32, device="cpu")
messages = [{"role": "user", "content": "Give me two tips for a job interview. Be brief."}]
out = chat(messages, max_new_tokens=40, do_sample=False)      # greedy: the same text every run
conversation = out[0]["generated_text"]                       # the input messages + the new assistant turn
print("roles:", [m["role"] for m in conversation])
print("assistant:", conversation[-1]["content"])
print(f"({time.perf_counter() - t0:.1f}s including the model load)")
assert conversation[-1]["role"] == "assistant"

A 135 M-parameter model gives fluent, generic advice and often stops mid-sentence at `max_new_tokens`. That is expected at this size. The point is the plumbing: the same three lines run a 7 B model on a GPU box.

### Which tasks have a pipeline?

Each task string maps to one pipeline class. The table lists the common ones; the typical models are examples you would download yourself, and none of them are downloaded here.

| Task string | Input → output | Typical model |
|---|---|---|
| `"text-classification"` (alias `"sentiment-analysis"`) | text → label + score | `distilbert/distilbert-base-uncased-finetuned-sst-2-english` |
| `"token-classification"` (alias `"ner"`) | text → a label per token/entity; use `aggregation_strategy="simple"` to merge word pieces | `dslim/bert-base-NER` |
| `"zero-shot-classification"` | text + labels you name at call time → scores (an NLI model scores "this text is about {label}") | `facebook/bart-large-mnli` |
| `"fill-mask"` | text with `[MASK]` → likely words | `google-bert/bert-base-uncased` |
| `"feature-extraction"` | text → per-token vectors | any encoder |
| `"text-generation"` | prompt or chat messages → continuation | SmolLM2, Qwen, Llama |
| `"automatic-speech-recognition"` | audio → text | `openai/whisper-small` |
| `"image-classification"` | image → labels | `google/vit-base-patch16-224` |
| `"zero-shot-image-classification"` | image + label texts → scores (CLIP) | `openai/clip-vit-base-patch32` |
| `"object-detection"` | image → boxes + labels | `facebook/detr-resnet-50` |
| `"image-text-to-text"` | image + prompt → text (vision-language models) | `Qwen/Qwen3-VL-2B-Instruct` |

**v4 → v5 gotcha:** transformers v5 removed the `summarization`, `translation`, `text2text-generation` and `question-answering` pipelines. Older tutorials that use them now fail with `KeyError: Unknown task`. Today you summarise or translate by prompting an instruct model through `"text-generation"`. Don't take a blog post's word for which tasks exist; ask the installed library:

In [ ]:
from transformers.pipelines import get_supported_tasks

tasks = get_supported_tasks()
print(len(tasks), "task strings:", ", ".join(tasks))
for gone in ["summarization", "translation", "text2text-generation", "question-answering"]:
    assert gone not in tasks, gone
assert {"text-classification", "ner", "zero-shot-classification", "text-generation"} <= set(tasks)

## 4. Downloading exactly what you need

`pipeline()` and `from_pretrained()` download for you. When you need a single file, or a whole repo for another runtime (vLLM, llama.cpp, a Docker image), use `huggingface_hub` directly:

| Function | Does | Typical use |
|---|---|---|
| `hf_hub_download(repo_id, filename, revision=...)` | one file → a local path in the cache | read a `config.json`, fetch one GGUF file |
| `snapshot_download(repo_id, allow_patterns=[...])` | a whole revision (optionally filtered) → a local folder | pre-bake a model into an image; skip the ONNX/TF copies |
| `local_files_only=True` (either function) | never touch the network | air-gapped servers, reproducible CI |

On the command line the same things are `hf download <repo> [files]`, `hf cache ls` and `hf cache rm`.

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download

repo = "HuggingFaceTB/SmolLM2-135M-Instruct"
cfg_path = hf_hub_download(repo, "config.json")
cfg = json.loads(Path(cfg_path).read_text())
print("local path:", cfg_path)
print({k: cfg[k] for k in ["model_type", "num_hidden_layers", "hidden_size", "vocab_size", "torch_dtype"]})

t0 = time.perf_counter()
folder = snapshot_download(repo, allow_patterns=["*.json", "*.txt"], local_files_only=True)   # cache only, no network
print(f"snapshot folder ({(time.perf_counter() - t0) * 1000:.0f} ms, no network):", folder)
print("json/txt files:", sorted(p.name for p in Path(folder).iterdir() if p.suffix in {".json", ".txt"}))

## 5. The cache: `HF_HOME` and `scan_cache_dir`

Everything goes into one cache, so a model is downloaded only once per machine:

```
~/.cache/huggingface/            <- HF_HOME (override with the env var)
└── hub/                         <- HF_HUB_CACHE
    └── models--HuggingFaceTB--SmolLM2-135M-Instruct/
        ├── refs/main            <- a text file holding the commit sha that "main" pointed to
        ├── snapshots/<sha>/     <- the files of that revision (symlinks into blobs/ on Linux/macOS;
        │                           plain copies on Windows without Developer Mode)
        ├── blobs/               <- content-addressed file contents, shared between revisions
        └── .no_exist/<sha>/     <- remembers files the repo does not have, so it doesn't keep asking
```

| Environment variable | Effect |
|---|---|
| `HF_HOME` | move the whole cache (for example to a big data disk) |
| `HF_HUB_OFFLINE=1` | never call the Hub; use only the cache (great for CI and air-gapped servers) |
| `HF_TOKEN` | your access token (section 8) |
| `HF_HUB_DISABLE_PROGRESS_BARS=1` | quiet logs in batch jobs |

In [ ]:
from huggingface_hub import scan_cache_dir, constants

print("HF_HOME      :", constants.HF_HOME)
print("HF_HUB_CACHE :", constants.HF_HUB_CACHE)
cache = scan_cache_dir()
print(f"cache holds {len(cache.repos)} repos, {cache.size_on_disk / 1e6:,.0f} MB in total\n")

rows = sorted(cache.repos, key=lambda r: r.size_on_disk, reverse=True)
for r in rows:
    print(f"{r.repo_type:8s} {r.repo_id:58s} {r.size_on_disk / 1e6:9.1f} MB  {len(r.revisions)} revision(s)  {r.nb_files} files")

fig, ax = plt.subplots(figsize=(9, 0.45 * len(rows) + 1))
ax.barh([r.repo_id.split("/")[-1][:34] for r in rows], [r.size_on_disk / 1e6 for r in rows], color="#4c6ef5")
ax.set_xscale("log"); ax.invert_yaxis()
ax.set_xlabel("size on disk (MB, log scale)"); ax.set_title("What is in your Hugging Face cache")
plt.show()

Sizes differ by orders of magnitude, which is why the x-axis is logarithmic. A tokenizer-only repo is a few MB, a small model a few hundred MB, and one cached dataset can be larger than all the models combined. Caches grow quietly on shared servers; `hf cache ls` shows the same table and `hf cache rm` deletes a repo.

## 6. Pin the revision you tested

`main` is a moving branch. If the author pushes new weights or a new tokenizer, your service changes behaviour on its next restart without any change in your code. Pin a **commit sha** and move to a new one on purpose, after re-running your evals. A sha is immutable, so a pinned load that is already cached needs no network at all.

In [ ]:
from transformers import AutoConfig

smol = next(r for r in cache.repos if r.repo_id == repo)
sha = next(iter(smol.revisions)).commit_hash
print("pinned commit:", sha)

t0 = time.perf_counter()
cfg_pinned = AutoConfig.from_pretrained(repo, revision=sha)    # same for from_pretrained / pipeline(revision=...)
print(f"AutoConfig at that commit: {cfg_pinned.model_type}, {cfg_pinned.num_hidden_layers} layers "
      f"({(time.perf_counter() - t0) * 1000:.0f} ms)")
assert cfg_pinned.hidden_size == cfg["hidden_size"]

commits = try_hub(api.list_repo_commits, repo)
if commits:
    print(f"\nthe repo has {len(commits)} commits; the latest few:")
    for c in commits[:4]:
        print(f"  {c.commit_id[:10]}  {str(c.created_at)[:10]}  {c.title[:60]}")

## 7. Security: pickle, safetensors and `trust_remote_code`

A model repo is **code plus weights from a stranger** (`sf27`). There are two ways it can run code on your machine.

**1. Pickle-based weight files** (`pytorch_model.bin`, `.pt`, `.ckpt`). Python's pickle format can store "call this function when loading", so loading the file can run arbitrary code. The demo below uses a harmless `print` in place of `os.system(...)`.

In [ ]:
class NotAWeight:
    """A pickle 'payload': unpickling calls print(...). An attacker would call os.system(...) instead."""
    def __reduce__(self):
        return (print, ("  >>> this ran while the file was being LOADED <<<",))

evil_file = Path(tempfile.mkdtemp()) / "pytorch_model.bin"
with open(evil_file, "wb") as f:
    pickle.dump({"layer.weight": [0.1, 0.2], "extra": NotAWeight()}, f, protocol=2)

print("plain pickle.load:")
with open(evil_file, "rb") as f:
    pickle.load(f)                      # the payload runs here

print("torch.load (weights_only=True is the default since PyTorch 2.6):")
try:
    torch.load(evil_file)
    raise AssertionError("torch.load should have refused this file")
except pickle.UnpicklingError as e:
    reason = next(line.strip() for line in str(e).splitlines() if "WeightsUnpickler error" in line)
    print("  refused ->", reason[:110])

**Safetensors** removes this risk by design. The file is an 8-byte header length, a JSON header (tensor names, dtypes, shapes, byte offsets), then raw numbers, so there is nothing to execute. It is also memory-mappable, which lets you read one tensor without loading the whole file. `from_pretrained` picks `model.safetensors` whenever a repo has it. `all-MiniLM-L6-v2`, for example, also ships `pytorch_model.bin`, `tf_model.h5` and ONNX files, which are never loaded.

In [ ]:
from safetensors import safe_open

weights_path = hf_hub_download(repo, "model.safetensors")
with open(weights_path, "rb") as f:
    header_len = struct.unpack("<Q", f.read(8))[0]          # little-endian uint64
    header = json.loads(f.read(header_len))
header.pop("__metadata__", None)
print(f"header: {header_len:,} bytes of JSON describing {len(header)} tensors, e.g.")
for name in list(header)[:2]:
    print("  ", name, header[name])

with safe_open(weights_path, framework="pt") as sf:        # memory-mapped: reads only what you ask for
    norm = sf.get_tensor("model.norm.weight")
    assert len(list(sf.keys())) == len(header)
print("read one tensor without loading the rest:", tuple(norm.shape), norm.dtype)

**2. `trust_remote_code=True`.** Some repos ship their own Python modelling files and list them under `auto_map` in `config.json`. Passing `trust_remote_code=True` tells transformers to **download and execute that Python code**. By default it refuses. If you need it:

- read the code first, and **pin the revision** (a sha), so the code you reviewed is the code that runs next week;
- prefer an architecture transformers supports natively (most popular ones get added within weeks);
- load it in a sandbox or container without secrets in the environment.

None of the models used in these notebooks need remote code:

In [ ]:
for rid in [repo, "sentence-transformers/all-MiniLM-L6-v2",
            "distilbert/distilbert-base-uncased-finetuned-sst-2-english", "cross-encoder/ms-marco-MiniLM-L6-v2"]:
    c = json.loads(Path(hf_hub_download(rid, "config.json")).read_text())
    print(f"{rid:60s} model_type={c['model_type']:11s} needs remote code: {'auto_map' in c}")
    assert "auto_map" not in c

## 8. Private and gated models, and `HF_TOKEN`

- **Private** repos are visible only to you or your organisation. **Gated** repos (Llama, some Gemma and Mistral models) are public but ask you to accept the licence on the website first. Both need a token.
- Create a token at huggingface.co → Settings → Access tokens. Use a **fine-grained, read-only** token for servers.
- Give it to the libraries in one of three ways: run `hf auth login` once (it stores the token in the cache folder), set the `HF_TOKEN` environment variable (containers and CI), or pass `token=...` to a single call.
- Never hard-code it in a notebook or commit it. A leaked write token lets someone push weights to your repos, which is its own supply-chain attack.

```python
# on a server: export HF_TOKEN=hf_xxx   (from your secret manager, not from code)
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")   # gated: token + accepted licence
```

In [ ]:
from huggingface_hub import get_token

print("a token is configured on this machine :", get_token() is not None)    # never print the token itself
print("HF_TOKEN environment variable is set :", bool(os.environ.get("HF_TOKEN")))

## 9. Hosted inference: Inference Providers and `InferenceClient`

Sometimes you want an open model without running it yourself. **Inference Providers** routes your request to a partner GPU cloud (Groq, Together, Fireworks, Novita, and others) with one HF token and one bill. You get some free credits each month, and after that you pay the provider's per-token price. The router speaks the OpenAI Chat Completions format, so either client works:

```python
import os
from huggingface_hub import InferenceClient

client = InferenceClient(provider="auto", api_key=os.environ["HF_TOKEN"])   # "auto" = first available provider
resp = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": "In one sentence, what is a tokenizer?"}],
    max_tokens=200,
)
print(resp.choices[0].message.content)

# the same call with the OpenAI SDK - only base_url and the key change
from openai import OpenAI
oai = OpenAI(base_url="https://router.huggingface.co/v1", api_key=os.environ["HF_TOKEN"])
```

For production you would use **Inference Endpoints** (a dedicated deployment of one model in the cloud region you choose), or serve it yourself with vLLM or TGI. Self-hosting pays off when volume is steady and high, or when data may not leave your network (`in19`).

The cell below makes that call only if you have set `HF_TOKEN` yourself, because it spends a fraction of a cent of your credits. Otherwise it just says it skipped.

In [ ]:
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import InferenceClient
    client = InferenceClient(provider="auto", api_key=os.environ["HF_TOKEN"])
    try:
        resp = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[{"role": "user", "content": "In one sentence, what is a tokenizer?"}],
            max_tokens=200)
        print(resp.choices[0].message.content)
    except Exception as e:
        print("Inference Providers call failed:", type(e).__name__, str(e)[:200])
else:
    print("HF_TOKEN is not set: skipped the remote call. Everything else in this notebook ran locally.")

## Try it yourself

**1. Add a "neutral" band.** The SST-2 model has no neutral class. Write `label_with_neutral(texts, margin=0.9)` that returns `"NEUTRAL"` when the top probability is below `margin`, and otherwise the model's label. Which of the five reviews become neutral?

In [ ]:
# Solution — try it yourself first, then run this
def label_with_neutral(texts, margin=0.9):
    out = []
    for r in sentiment(texts):
        out.append(r["label"] if r["score"] >= margin else "NEUTRAL")
    return out

labs = label_with_neutral(reviews)
for text, lab in zip(reviews, labs):
    print(f"{lab:>8}  {text}")
assert labs[3] == "NEUTRAL"      # the sarcastic review: the model was unsure, so the band catches it
assert labs[4] != "NEUTRAL"      # the flat "package arrived" review: the model is (wrongly) very sure
# A threshold only helps where the model is unsure. Fixing confident mistakes needs a model trained with a neutral class.

**2. Read the chat template from the cache.** Use `hf_hub_download` with `local_files_only=True` to fetch SmolLM2's `tokenizer_config.json`, then print the first 150 characters of its `chat_template` field.

In [ ]:
# Solution — try it yourself first, then run this
tc = json.loads(Path(hf_hub_download(repo, "tokenizer_config.json", local_files_only=True)).read_text())
print(tc["chat_template"][:150], "...")
assert "<|im_start|>" in tc["chat_template"]

**3. Cache size per repo.** Write `cached_mb(repo_id)` that uses `scan_cache_dir()` and returns the size of one cached repo in MB (or `0.0` if it is not cached). Check that `sentence-transformers/all-MiniLM-L6-v2` is under 200 MB.

In [ ]:
# Solution — try it yourself first, then run this
def cached_mb(repo_id):
    for r in scan_cache_dir().repos:
        if r.repo_id == repo_id:
            return r.size_on_disk / 1e6
    return 0.0

mb = cached_mb("sentence-transformers/all-MiniLM-L6-v2")
print(f"all-MiniLM-L6-v2 uses {mb:.1f} MB; a repo that is not cached: {cached_mb('not/cached'):.1f} MB")
assert 0 < mb < 200

## Say it in an interview

- **"What is Hugging Face?"** "Three things. The Hub is git-based hosting for models, datasets and Spaces. The libraries are transformers, datasets, sentence-transformers, PEFT, TRL and accelerate. Then there's hosted inference: Inference Providers and Endpoints. In practice it's the package manager for open models." Then add the risk: "A repo is code plus weights from a stranger, so I load only safetensors, pin a commit sha, and don't set `trust_remote_code` unless I've reviewed the code."
- **Choosing a model:** task and licence first, then size (parameters × bytes = memory), base or instruct, safetensors available, how active the repo is. Popularity is a hint, never the evaluation.
- **`pipeline()` inside:** tokenizer → model forward → post-processing (softmax + `id2label` for classifiers, decoding for generation). You can reproduce it by hand in five lines, as section 3 did.
- **Scores are not truth:** a binary sentiment model gives neutral text a confident label, and sarcasm flips it. Thresholds need a model that knows it is unsure.
- **Reproducibility:** `revision=<sha>`, `HF_HUB_OFFLINE=1` and a pre-filled `HF_HOME` in the image. Check the licence before building, since `cc-by-nc` blocks commercial use.
- **Traps:** transformers v5 removed the summarisation/translation/QA pipelines and changed the default dtype to `"auto"`, so load in bf16/fp32 on purpose. A PyTorch `.bin` file is a pickle and can run code on load. Never print or commit `HF_TOKEN`.
- **Likely follow-ups:** "how would you serve this in production?" (vLLM/TGI, or Inference Endpoints), "how do you run it air-gapped?" (`snapshot_download` at build time + `HF_HUB_OFFLINE=1`), "what does `device_map='auto'` do?" (accelerate spreads the layers over GPUs, CPU and disk).

## Next

- [02 · Tokenizers, models and `generate()`](02_tokenizers_models_and_generate.ipynb): what the pipeline hides.
- [03 · The datasets library](03_datasets_library.ipynb) · [04 · Sentence-transformers](04_sentence_transformers_embeddings_and_cross_encoders.ipynb) · [05 · Fine-tune with the Trainer](05_trainer_api_fine_tune_small_model.ipynb) · [06 · Quantization with bitsandbytes](06_quantization_with_bitsandbytes.ipynb)
- Serving the model yourself: [Local LLMs with Ollama](../08_llm_apps/05_local_llms_with_ollama.ipynb), [FastAPI serving and streaming](../16_production/04_fastapi_serving_and_streaming.ipynb)
- Theory: [genai_flow course](../../genai_flow/index.html), [ai_tooling course](../../ai_tooling/index.html), [interview bank](../../ai_interview_prep/index.html)